# From Model Scores to Governed Decisions: Complete Colab Lab

This lab is the executable companion notebook for the article series **From Model Scores to Governed Decisions**.

It walks through the complete end-to-end workflow:

1. Load validation labels and model scores.
2. Convert model scores into decisions using thresholds.
3. Evaluate baseline model behavior.
4. Sweep candidate thresholds.
5. Add business value assumptions.
6. Add operational capacity constraints.
7. Select a governed threshold with guardrails.
8. Compare default and selected thresholds.
9. Apply segment-specific thresholds.
10. Create auditable production decision logs.
11. Generate stakeholder-ready charts.

The goal is to show that a binary classification model does not create a business decision by itself. The threshold does.

## 0. Colab Setup

Run this cell first in Google Colab. It clones the repository if needed, installs the package in editable mode, and moves into the repo folder.

If you are running this notebook locally from the repository root, the same cell will use the current checkout.

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = 'https://github.com/shalabhdixit/from-model-scores-to-governed-decisions.git'
REPO_DIR = Path('/content/from-model-scores-to-governed-decisions')
LOCAL_MARKER = Path('pyproject.toml')

if 'google.colab' in sys.modules:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
elif LOCAL_MARKER.exists():
    print('Using local repository checkout:', Path.cwd())
else:
    print('Run this notebook from the repository root, or use Google Colab so the repo can be cloned automatically.')

!python -m pip install --quiet --upgrade pip
!python -m pip install --quiet -e .[dev]

print('Repository ready at:', Path.cwd())

## 1. Import The Reusable Package

The repository is structured as a real Python package. The notebook uses the same reusable modules that the scripts and tests use.

In [ ]:
import pandas as pd

from governed_decisions.business_value import DEFAULT_BUSINESS_VALUES
from governed_decisions.decision_logging import score_decision
from governed_decisions.metrics import evaluate_threshold
from governed_decisions.reporting import compare_thresholds, select_threshold, threshold_evaluation_report
from governed_decisions.sample_data import load_sample_validation_data, sample_arrays
from governed_decisions.segment_thresholds import apply_segment_thresholds
from governed_decisions.thresholding import apply_threshold
from governed_decisions.visualization import plot_business_value, plot_metric_tradeoffs

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

## 2. Load Validation Data

A threshold can only be evaluated when you have observed outcomes and model probability scores. This sample dataset uses three fields:

| Column | Meaning |
| --- | --- |
| `actual` | Observed binary outcome. `1` is the positive class. |
| `score` | Model probability score for the positive class. |
| `segment` | Optional segment used for contextual thresholds. |

In [ ]:
data = load_sample_validation_data()
actual_labels, model_scores = sample_arrays()

print(data.head())
print('Records:', len(data))

## 3. Apply A Baseline Threshold

The default `0.50` threshold is common, but it is only a starting point. It converts scores into class labels.

In [ ]:
baseline_threshold = 0.50
data['prediction_at_050'] = apply_threshold(data['score'], threshold=baseline_threshold)
data[['actual', 'score', 'prediction_at_050']].head(10)

## 4. Evaluate The Baseline Threshold

This step computes the metrics that matter for the starting point: confusion matrix counts, precision, recall, F1, accuracy, predicted positive rate, and ROC-AUC.

In [ ]:
baseline_metrics = evaluate_threshold(actual_labels, model_scores, threshold=baseline_threshold)
pd.Series(baseline_metrics)

## 5. Sweep Candidate Thresholds

A real decision policy should compare several candidate operating points. This report evaluates thresholds from `0.10` to `0.90`.

In [ ]:
report_without_capacity = threshold_evaluation_report(actual_labels, model_scores)
report_without_capacity[[
    'threshold', 'accuracy', 'precision', 'recall', 'f1',
    'true_positives', 'false_positives', 'false_negatives', 'predicted_positive_rate',
]]

## 6. Add Business Value

Technical metrics count errors. Business value estimates the consequence of those errors. The default assumptions are intentionally simple and should be replaced with real finance, risk, product, and operations inputs in production.

In [ ]:
DEFAULT_BUSINESS_VALUES

In [ ]:
report_without_capacity.sort_values('business_value', ascending=False)[[
    'threshold', 'precision', 'recall', 'f1', 'false_positives', 'false_negatives', 'business_value'
]].head(8)

## 7. Add Operational Capacity

Many classification systems route cases to a human team. A threshold that creates too much review volume may be statistically attractive but operationally unusable.

In [ ]:
max_manual_reviews = 12
report = threshold_evaluation_report(actual_labels, model_scores, max_manual_reviews=max_manual_reviews)
report[[
    'threshold', 'precision', 'recall', 'f1', 'manual_reviews', 'within_capacity', 'business_value'
]]

## 8. Select The Threshold With Guardrails

The selected threshold must satisfy business and operating guardrails. In this lab, the threshold must meet minimum recall, minimum precision, and manual review capacity constraints.

In [ ]:
selected = select_threshold(
    report,
    min_recall=0.70,
    min_precision=0.60,
    require_capacity=True,
)
selected[['threshold', 'precision', 'recall', 'f1', 'manual_reviews', 'within_capacity', 'business_value']]

## 9. Compare Default And Selected Thresholds

Stakeholders need to understand what changes when the operating threshold changes. This comparison makes the tradeoff explicit.

In [ ]:
comparison = compare_thresholds(report, baseline_threshold=0.50, selected_threshold=float(selected['threshold']))
comparison

## 10. Visualize Threshold Tradeoffs

The charts show two different conversations: model behavior and business consequence.

In [ ]:
from IPython.display import Image, display

metric_chart = plot_metric_tradeoffs(report, 'outputs/metric-tradeoffs.png')
business_chart = plot_business_value(report, 'outputs/business-value.png', selected_threshold=float(selected['threshold']))

display(Image(filename=str(metric_chart)))
display(Image(filename=str(business_chart)))

## 11. Apply Segment-Specific Thresholds

Segment-aware thresholds can improve operational fit, but they also create governance obligations. In production, each segment threshold should have rationale, approval, monitoring, and fairness review.

In [ ]:
segment_thresholds = {
    'high_value': 0.65,
    'standard': 0.45,
}

segmented_data = apply_segment_thresholds(data, segment_thresholds)
segmented_data[['actual', 'score', 'segment', 'segment_threshold', 'segment_prediction']].head(12)

## 12. Create A Production Decision Record

A production decision record should preserve the score, threshold, predicted label, route, model version, threshold version, and timestamp. This is the foundation for auditability and rollback.

In [ ]:
decision = score_decision(
    record_id='TXN-10001',
    model_score=0.62,
    threshold=float(selected['threshold']),
    model_version='fraud-model-v4',
    threshold_version='threshold-policy-2026-05',
)
decision

## 13. Governance Checklist

Before a threshold becomes production policy, the team should answer these questions:

| Governance Question | Why It Matters |
| --- | --- |
| Which model version produced the score? | Supports model lineage and incident investigation. |
| Which threshold version was active? | Supports policy audit and rollback. |
| What business value assumptions were used? | Prevents hidden cost and risk assumptions. |
| Did the policy fit operational capacity? | Prevents queue overload and SLA failure. |
| Were segment thresholds reviewed for fairness? | Prevents unmanaged disparate impact. |
| Who approved the threshold? | Makes ownership explicit. |
| How will drift and overrides be monitored? | Keeps the decision policy healthy after deployment. |

## 14. Final Takeaway

The model estimates probability. The threshold defines behavior. Governance determines whether that behavior can be trusted.

A production AI system should not only ask whether the model is accurate. It should ask whether the decision policy is valuable, operable, explainable, monitored, and governed.